# Copy one legacy forecast configuration into the v1 `public` project

Copies a forecast configuration from the legacy warehouse (`iceberg.teehr`, the pre-v1
model) into `public` as a v1 forecast dataset (`<CONFIGURATION>`, from
`secondary_timeseries`), so the control plane has real forecasts to evaluate.

The gages, their observations and the crosswalk to this configuration's reaches come from
**`05_rebuild_usgs`**: run it after this one (it copies the crosswalk once the forecast
exists).

**Run it as a platform admin** (the dataset is registered in `public` through the control
plane API, as you). Works on KinD (its legacy warehouse) and use1.

**Spark writes the rows straight into the `public` Iceberg table**, not through the API's
write path: a configuration is hundreds of millions of rows, and the API takes at most
64 MB per write. So these writes skip the API's write lineage (the months a write
touched); the first run of an evaluation over them is a full run anyway. A one-time admin
migration, not a pattern (ADR-0004).

**Re-runnable:** registering skips a dataset that exists; the series table is replaced
whole; the forecasts are written a month at a time (of `reference_time`), each replacing
that month's partition. Run again, or from a month on, any time the legacy data changes.

Start with **2. Count**: it writes nothing.

In [ ]:
import os

import requests
import teehr
from pyspark.sql import functions as F
from teehr.evaluation.spark_session_utils import create_spark_session

teehr.__version__

## 1. Settings

In [ ]:
CONFIGURATION = "nwm30_medium_range"

# The control plane API, from inside the cluster (KinD and use1 alike). From outside:
# https://control-plane.teehr.rtiamanzi.org (use1), https://control-plane.teehr.local.app.garden (KinD).
CONTROL_PLANE = os.environ.get(
    "CONTROL_PLANE_URL", "http://teehr-control-plane.teehr-hub.svc.cluster.local:8000"
)
VERIFY_TLS = "local.app.garden" not in CONTROL_PLANE  # KinD's certificate is self-signed

LEGACY = "iceberg.teehr"
PUBLIC = "iceberg.public"
FORECAST = CONFIGURATION

In [ ]:
# Local mode for counting and small copies; for a whole configuration start a cluster,
# e.g. create_spark_session(start_spark_cluster=True, executor_instances=32,
#                           executor_memory="16g", executor_cores=4)
spark = create_spark_session(
    # start_spark_cluster=True,
    # executor_instances=4,
    # executor_memory="16g",
    # executor_cores=4
)
# The v1 tables declare required columns; the legacy columns are nullable in Spark's eyes.
spark.conf.set("spark.sql.iceberg.check-nullability", "false")
# Months are UTC everywhere (Iceberg's month partitions are): without this a
# "month" in the session's zone straddles two partitions and a re-run could drop rows.
spark.conf.set("spark.sql.session.timeZone", "UTC")


## 2. Count (writes nothing)

From Iceberg's partition metadata (`<table>.partitions`: rows per partition, no scan)
when the legacy table is partitioned by `configuration_name` and a time field; otherwise
by scanning. Then a look at one month's units, variables and members.

In [ ]:
%%time
from datetime import date, datetime, timedelta

import pandas as pd


def month_of(field, value):
    """The month (YYYY-MM) of a partition value of an Iceberg time transform."""
    if value is None:
        return None
    if isinstance(value, (date, datetime)):
        return f"{value:%Y-%m}"
    if field.endswith("_month"):  # months since 1970-01
        return f"{1970 + value // 12:04d}-{value % 12 + 1:02d}"
    if field.endswith("_day"):
        return f"{date(1970, 1, 1) + timedelta(days=value):%Y-%m}"
    if field.endswith("_hour"):
        return f"{datetime(1970, 1, 1) + timedelta(hours=value):%Y-%m}"
    return None  # a year partition: coarser than a month


def counts(table, configuration):
    """Rows per month of one configuration, and the time column the months are of."""
    parts = spark.table(f"{LEGACY}.{table}.partitions")
    fields = parts.schema["partition"].dataType.fieldNames()
    print(f"{table} partitioned by: {fields}")
    config = next((f for f in fields if f.startswith("configuration_name")), None)
    time = next((f for f in fields if f.startswith(("value_time", "reference_time"))), None)
    if config and time and not time.endswith("_year"):
        rows = (
            parts.where(F.col(f"partition.{config}") == configuration)
            .select(F.col(f"partition.{time}").alias("t"), "record_count")
            .toPandas()
        )
        rows["month"] = [month_of(time, t) for t in rows.t]
        out = rows.groupby("month", as_index=False).record_count.sum()
        out = out.rename(columns={"record_count": "rows"}).sort_values("month")
        return out.reset_index(drop=True), time.rsplit("_", 1)[0]
    print(f"  not partitioned by configuration and time: scanning {table}")
    out = spark.sql(f"""
        SELECT date_format(value_time, 'yyyy-MM') AS month, count(*) AS rows
        FROM {LEGACY}.{table} WHERE configuration_name = '{configuration}'
        GROUP BY 1 ORDER BY 1
    """).toPandas()
    return out, "value_time"


forecasts, forecast_time = counts("secondary_timeseries", CONFIGURATION)
print(f"{forecasts.rows.sum():,} forecast rows in {len(forecasts)} months of {forecast_time}")
forecasts

In [ ]:
# Every month the forecasts span.
MONTHS = [
    str(m)  # YYYY-MM
    for m in pd.period_range(forecasts.month.min(), forecasts.month.max(), freq="M")
]
print(f"{len(MONTHS)} months, {MONTHS[0]} .. {MONTHS[-1]}")

In [ ]:
def month_range(column, month):
    """A time range filter for one UTC month (prunes the scan, unlike date_format)."""
    start = pd.Period(month, freq="M")
    end = start + 1
    return (f"{column} >= TIMESTAMP '{start.start_time:%Y-%m-%d} 00:00:00' "
            f"AND {column} < TIMESTAMP '{end.start_time:%Y-%m-%d} 00:00:00'")


# Units, variables and members, from one month (enough to check them).
spark.sql(f"""
    SELECT unit_name, variable_name, member, count(*) AS rows
    FROM {LEGACY}.secondary_timeseries
    WHERE configuration_name = '{CONFIGURATION}' AND {month_range("value_time", MONTHS[0])}
    GROUP BY 1, 2, 3 ORDER BY 1, 2, 3
""").show(truncate=False)

## 3. Register the v1 dataset in `public` (as you)

Left as it is if it already exists. Values are streamflow in m3/s (the legacy `m^3/s`):
the count above must show only that unit.

In [ ]:
def access_token():
    """Your access token, refreshed when it's about to expire (JupyterHub sets it)."""
    try:
        from teehr.utils.keycloak_auth import refresh_user_token_from_env
    except ImportError:  # a kernel without the helper: the token as JupyterHub set it
        return os.environ["POLARIS_USER_TOKEN"]
    access, _refresh = refresh_user_token_from_env()
    return access


api = requests.Session()
api.verify = VERIFY_TLS


def call(method, path, **kwargs):
    resp = api.request(method, f"{CONTROL_PLANE}/v1{path}",
                       headers={"Authorization": f"Bearer {access_token()}"}, **kwargs)
    if resp.status_code >= 400 and resp.status_code != 404:
        raise RuntimeError(f"{method} {path}: HTTP {resp.status_code}: {resp.text[:1000]}")
    return resp


def register(body):
    path = f"/projects/public/datasets/{body['name']}"
    if call("GET", path).status_code == 200:
        print(f"public.{body['name']}: exists")
        return
    call("POST", "/projects/public/datasets", json=body)
    print(f"public.{body['name']}: registered")


me = call("GET", "/projects/public").json()
print(f"public: {me['status']}; your role there: {me.get('my_role')}")


members = spark.sql(f"""
    SELECT count(*) AS n FROM {LEGACY}.secondary_timeseries
    WHERE configuration_name = '{CONFIGURATION}' AND member IS NOT NULL
      AND {month_range("value_time", MONTHS[0])}
""").first().n > 0
source = "Legacy TEEHR warehouse (copied)"
register({"type": "timeseries", "name": FORECAST, "role": "forecast",
          "variable": "streamflow", "statistic": "inst",
          "optional_columns": ["member"] if members else [],
          "source": source, "description": f"{CONFIGURATION} (legacy copy)"})

## 4. Copy

The series table first (small, replaced whole), then the forecasts a month at a time.

In [ ]:
%%time
spark.sql(f"""
    SELECT DISTINCT location_id AS series_key, CAST(NULL AS STRING) AS location_key,
           CAST(NULL AS STRING) AS location_layer, CAST(NULL AS STRING) AS period,
           CAST(NULL AS STRING) AS resolution
    FROM {LEGACY}.secondary_timeseries WHERE configuration_name = '{CONFIGURATION}'
""").writeTo(f"{PUBLIC}.{FORECAST}_series").overwrite(F.lit(True))
print("series table copied")

In [ ]:
def copy_month(month):
    """Replace one UTC month (of reference_time) of the forecasts with the legacy rows."""
    # The legacy table is partitioned by month of value_time: bound value_time too (a
    # forecast is valid from its issue to its longest lead), so the scan reads two months
    # of partitions, not all.
    start = pd.Period(month, freq="M").start_time
    end = (pd.Period(month, freq="M") + 1).start_time + pd.Timedelta(MAX_LEAD)
    rows = spark.sql(f"""
        SELECT {", ".join(FORECAST_COLUMNS)} FROM {LEGACY}.secondary_timeseries
        WHERE configuration_name = '{CONFIGURATION}' AND {month_range("reference_time", month)}
          AND value_time >= TIMESTAMP '{start:%Y-%m-%d} 00:00:00'
          AND value_time < TIMESTAMP '{end:%Y-%m-%d %H:%M:%S}'
    """)
    rows.writeTo(f"{PUBLIC}.{FORECAST}").overwritePartitions()


FORECAST_COLUMNS = ["location_id AS series_key", "reference_time", "value_time",
                    *(["member"] if members else []), "CAST(value AS DOUBLE) AS value"]

START = None  # e.g. "2025-03" to resume from a month
MAX_LEAD = "16D"  # longer than the configuration's longest lead (medium range: 10 days)

In [ ]:
%%time
for month in MONTHS:  # by month of reference_time (the v1 table's partitions)
    if START and month < START:
        continue
    copy_month(month)
    print(f"{FORECAST} {month}: copied")

## 5. Check: v1 rows against the legacy

Forecasts per month against the metadata counts (every row is copied). Then run
`05_rebuild_usgs` for the gages, observations and crosswalk.

In [ ]:
%%time
v1 = spark.sql(f"""
    SELECT date_format({forecast_time}, 'yyyy-MM') AS month, count(*) AS v1_rows
    FROM {PUBLIC}.{FORECAST} GROUP BY 1
""").toPandas()
both = forecasts.merge(v1, on="month", how="outer").fillna(0)
both["matches"] = both.rows == both.v1_rows
print(f"{FORECAST}: {int(both.v1_rows.sum()):,} rows; every month matches: {both.matches.all()}")
display(both[~both.matches])